# Demo 09 - Lateral movement graph

**Fast** (top-N subgraph) · **Pool:** Medium · **Visual:** directed network graph

**The question:** which machines are the hubs an attacker would pivot through?

Every remote logon becomes an arrow from one machine to another. Draw all of them and you
have a map of how the estate actually connects, with the busiest junctions sized and
coloured so they cannot be missed.

KQL can list the logons as rows. It cannot build a graph from them, score which machines
are central, or lay the result out so the shape is obvious.

## 1. Connect to the data lake

`MicrosoftSentinelProvider` is the bridge between this notebook and your lake. The `spark`
session is handed to you by the Microsoft Sentinel kernel, so never create your own.

Nothing is read yet. This cell only opens the connection.

In [ ]:
from sentinel_lake.providers import MicrosoftSentinelProvider
from pyspark.sql import functions as F
data_provider = MicrosoftSentinelProvider(spark)

## 2. Settings you can change

- `LOOKBACK_DAYS` - how much logon history to map.
- `TOP_EDGES` - how many device-to-device connections to draw. Sixty keeps the picture
  readable. Push it much higher and it turns into a hairball.

In [ ]:
WORKSPACE = "your-workspace-name"
LOOKBACK_DAYS = 7
TOP_EDGES = 60

## 3. Turn remote logons into a list of connections

Every row in `DeviceLogonEvents` that carries a `RemoteDeviceName` is one machine logging
in to another. That is an **edge**: source device to destination device, weighted by how
many times it happened.

We keep only the busiest `TOP_EDGES` pairs. Rows missing a device name at either end are
dropped, because they would become a nameless node and break the drawing.

In [ ]:
import pandas as pd

logon = data_provider.read_table("DeviceLogonEvents", WORKSPACE)
edges = (logon.filter(F.col("TimeGenerated") >= F.expr(f"current_timestamp() - INTERVAL {int(LOOKBACK_DAYS)} DAYS"))
              .filter(F.col("RemoteDeviceName").isNotNull() & (F.col("RemoteDeviceName") != ""))
              # a null DeviceName would become a None node, and networkx hands node labels
              # straight to matplotlib text
              .filter(F.col("DeviceName").isNotNull() & (F.col("DeviceName") != ""))
              .groupBy(F.col("RemoteDeviceName").alias("src"), F.col("DeviceName").alias("dst"))
              .agg(F.count("*").alias("w"))
              .orderBy(F.desc("w")).limit(TOP_EDGES)).toPandas()
print("edges:", len(edges))
edges.head()

## 4. Build the graph and find the hubs

`networkx` turns that edge list into a real directed graph: machines are nodes, logons are
arrows pointing from source to destination.

**Degree centrality** is the score used to size the nodes. In plain terms: what share of
all the other machines in the picture does this one connect to? A machine that talks to
half the estate scores high; a laptop that only ever reaches one file server scores low.

The five highest-scoring machines are drawn in red and labelled. Node size follows the same
score, and edge thickness follows how many logons that pair saw.

**What to look for:** red hubs that *should* be hubs - jump boxes, admin workstations,
backup servers - are expected and fine. A red hub that turns out to be somebody's laptop is
the finding. So is a long thin chain of arrows hopping machine to machine, which is what
lateral movement actually looks like once you draw it.

In [ ]:
import networkx as nx, matplotlib.pyplot as plt

if edges.empty:
    print(f"No remote-logon edges in the last {LOOKBACK_DAYS} days - raise LOOKBACK_DAYS.")
else:
    G = nx.from_pandas_edgelist(edges, "src", "dst", edge_attr="w", create_using=nx.DiGraph())
    cent = nx.degree_centrality(G)
    sizes = [3000*cent[n] + 120 for n in G.nodes()]
    hubs = sorted(cent, key=cent.get, reverse=True)[:5]
    colors = ["#e74c3c" if n in hubs else "#3498db" for n in G.nodes()]

    plt.figure(figsize=(12, 9))
    pos = nx.spring_layout(G, k=.6, seed=42)
    nx.draw_networkx_edges(G, pos, alpha=.3, arrows=True, arrowsize=8,
                           width=[0.5+0.15*G[u][v]["w"] for u,v in G.edges()])
    nx.draw_networkx_nodes(G, pos, node_size=sizes, node_color=colors, alpha=.9)
    nx.draw_networkx_labels(G, pos, {n:n for n in hubs}, font_size=8, font_color="black")
    plt.title("Remote-logon graph - red = highest-centrality hub devices")
    plt.axis("off"); plt.tight_layout(); plt.show()
    print("Top hub devices:", hubs)

## Why a notebook beats KQL here

Building a directed graph, computing degree centrality, and laying it out visually are native to `networkx`. KQL can list edges as rows but can't traverse, score centrality, or draw the pivot map that makes lateral movement obvious.